In [1]:
!pip install pandas numpy faker

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------------------------------ --------- 1.6/2.1 MB 12.5 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 8.3 MB/s  0:00:00


In [2]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

In [3]:
fake = Faker()
np.random.seed(42)
random.seed(42)

NUM_PRODUCTS = 500
NUM_SUPPLIERS = 30
NUM_WAREHOUSES = 10
NUM_ORDERS = 100000

In [4]:
categories = ["Laptop", "Phone", "Accessories", "Storage", "Networking"]

products = []

for i in range(1, NUM_PRODUCTS + 1):
    cost = np.random.randint(100, 1000)
    margin = np.random.uniform(1.2, 1.6)

    products.append([
        f"P{i:03}",
        f"Product_{i}",
        random.choice(categories),
        cost,
        round(cost * margin, 2)
    ])

products_df = pd.DataFrame(
    products,
    columns=[
        "ProductID",
        "ProductName",
        "Category",
        "UnitCost",
        "SellingPrice"
    ]
)

products_df.head()

,ProductID,ProductName,Category,UnitCost,SellingPrice
0,P001,Product_1,Laptop,202,306.76
1,P002,Product_2,Laptop,370,552.34
2,P003,Product_3,Accessories,800,1150.99
3,P004,Product_4,Phone,221,278.99
4,P005,Product_5,Phone,430,594.99


In [5]:
countries = ["India", "China", "Germany", "USA"]

suppliers = []

for i in range(1, NUM_SUPPLIERS + 1):
    suppliers.append([
        f"S{i:03}",
        f"Supplier_{i}",
        random.choice(countries),
        np.random.randint(5, 20)
    ])

suppliers_df = pd.DataFrame(
    suppliers,
    columns=[
        "SupplierID",
        "SupplierName",
        "Country",
        "ContractLeadTime"
    ]
)

suppliers_df.head()

,SupplierID,SupplierName,Country,ContractLeadTime
0,S001,Supplier_1,China,15
1,S002,Supplier_2,India,18
2,S003,Supplier_3,Germany,14
3,S004,Supplier_4,Germany,12
4,S005,Supplier_5,India,10


In [6]:
regions = ["North", "South", "East", "West"]

warehouses = []

for i in range(1, NUM_WAREHOUSES + 1):
    warehouses.append([
        f"W{i:03}",
        f"Warehouse_{i}",
        random.choice(regions),
        np.random.randint(30000, 70000)
    ])

warehouses_df = pd.DataFrame(
    warehouses,
    columns=[
        "WarehouseID",
        "WarehouseName",
        "Region",
        "Capacity"
    ]
)

warehouses_df

,WarehouseID,WarehouseName,Region,Capacity
0,W001,Warehouse_1,South,68727
1,W002,Warehouse_2,North,30126
2,W003,Warehouse_3,West,57632
3,W004,Warehouse_4,South,50530
4,W005,Warehouse_5,West,69964
5,W006,Warehouse_6,East,33368
6,W007,Warehouse_7,West,33797
7,W008,Warehouse_8,West,43718
8,W009,Warehouse_9,West,64560
9,W010,Warehouse_10,South,41053


In [7]:
start_date = datetime(2024, 1, 1)
orders = []

for i in range(1, NUM_ORDERS + 1):
    product_id = f"P{np.random.randint(1, NUM_PRODUCTS+1):03}"
    supplier_id = f"S{np.random.randint(1, NUM_SUPPLIERS+1):03}"
    warehouse_id = f"W{np.random.randint(1, NUM_WAREHOUSES+1):03}"

    day_offset = np.random.randint(0, 730)
    order_date = start_date + timedelta(days=day_offset)

    quantity = np.random.randint(1, 20)

    if order_date.month in [11, 12]:
        quantity *= 2

    price = products_df.loc[
        products_df["ProductID"] == product_id,
        "SellingPrice"
    ].values[0]

    revenue = quantity * price

    orders.append([
        f"O{i:06}",
        order_date,
        product_id,
        supplier_id,
        warehouse_id,
        quantity,
        revenue
    ])

orders_df = pd.DataFrame(
    orders,
    columns=[
        "OrderID",
        "OrderDate",
        "ProductID",
        "SupplierID",
        "WarehouseID",
        "Quantity",
        "Revenue"
    ]
)

orders_df.head()

,OrderID,OrderDate,ProductID,SupplierID,WarehouseID,Quantity,Revenue
0,O000001,2025-09-13,P101,S021,W010,4,4382.84
1,O000002,2024-01-01,P182,S023,W009,19,23051.56
2,O000003,2025-03-06,P424,S015,W005,9,5985.18
3,O000004,2025-03-11,P309,S009,W010,9,7393.86
4,O000005,2025-08-24,P291,S020,W009,11,3457.74


In [8]:
shipments = []

for idx, row in orders_df.iterrows():
    ship_date = row["OrderDate"] + timedelta(days=np.random.randint(1, 4))

    if row["SupplierID"] == "S007":
        delay = np.random.randint(3, 10)
    else:
        delay = np.random.randint(-1, 4)

    delivery_date = ship_date + timedelta(days=7 + max(delay, 0))

    shipments.append([
        f"SH{idx+1:06}",
        row["OrderID"],
        ship_date,
        delivery_date,
        delay
    ])

shipments_df = pd.DataFrame(
    shipments,
    columns=[
        "ShipmentID",
        "OrderID",
        "ShipDate",
        "DeliveryDate",
        "DelayDays"
    ]
)

shipments_df.head()

,ShipmentID,OrderID,ShipDate,DeliveryDate,DelayDays
0,SH000001,O000001,2025-09-15,2025-09-25,3
1,SH000002,O000002,2024-01-03,2024-01-12,2
2,SH000003,O000003,2025-03-08,2025-03-18,3
3,SH000004,O000004,2025-03-14,2025-03-23,2
4,SH000005,O000005,2025-08-25,2025-09-02,1


In [10]:
inventory = []

for _, p in products_df.iterrows():
    for _, w in warehouses_df.iterrows():
        stock = np.random.randint(0, 500)

        if w["WarehouseID"] == "W003":
            stock += 300

        reorder = np.random.randint(50, 150)
        safety = np.random.randint(20, 80)

        potential_revenue = stock * p["SellingPrice"]

        inventory.append([
            p["ProductID"],
            w["WarehouseID"],
            stock,
            reorder,
            safety,
            round(potential_revenue, 2)
        ])

inventory_df = pd.DataFrame(
    inventory,
    columns=[
        "ProductID",
        "WarehouseID",
        "StockLevel",
        "ReorderPoint",
        "SafetyStock",
        "PotentialRevenue"
    ]
)

inventory_df.head()

,ProductID,WarehouseID,StockLevel,ReorderPoint,SafetyStock,PotentialRevenue
0,P001,W001,381,113,21,116875.56
1,P001,W002,437,70,45,134054.12
2,P001,W003,784,101,40,240499.84
3,P001,W004,237,68,30,72702.12
4,P001,W005,496,103,67,152152.96


In [12]:
products_df.to_csv("products.csv", index=False)
suppliers_df.to_csv("suppliers.csv", index=False)
warehouses_df.to_csv("warehouses.csv", index=False)
orders_df.to_csv("orders.csv", index=False)
shipments_df.to_csv("shipments.csv", index=False)
inventory_df.to_csv("inventory.csv", index=False)

print("All CSV files exported successfully!")

All CSV files exported successfully!


In [13]:
print("Products:", len(products_df))
print("Suppliers:", len(suppliers_df))
print("Warehouses:", len(warehouses_df))
print("Orders:", len(orders_df))
print("Shipments:", len(shipments_df))
print("Inventory:", len(inventory_df))

Products: 500
Suppliers: 30
Warehouses: 10
Orders: 100000
Shipments: 100000
Inventory: 5000
